# 03 — Sentiment Inference

Goal: run a pretrained Arabic sentiment model on the cleaned dataset and save predictions.

In this notebook we will:
- Load cleaned data from `data/processed/tweets_clean.csv`
- Run Hugging Face sentiment inference in batches
- Normalize model labels to `positive/negative/neutral`
- Validate on a random sample and compute quick accuracy vs existing labels
- Save output to `data/processed/tweets_with_sentiment.csv`

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import torch
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv('../data/processed/tweets_clean.csv')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(5)

In [ ]:
MODEL_NAME = 'CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment'
DEVICE = 0 if torch.cuda.is_available() else -1

sentiment_pipe = pipeline(
    'sentiment-analysis',
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=DEVICE,
)

print('Model loaded:', MODEL_NAME)
print('Device:', 'GPU' if DEVICE == 0 else 'CPU')
print('id2label:', sentiment_pipe.model.config.id2label)

In [ ]:
def normalize_label(raw_label: str, id2label: dict) -> str:
    label = str(raw_label)

    # If output is LABEL_0 style, map using id2label
    if label.startswith('LABEL_'):
        try:
            idx = int(label.split('_')[-1])
            label = id2label.get(idx, label)
        except Exception:
            pass

    l = label.lower()
    if 'pos' in l:
        return 'positive'
    if 'neg' in l:
        return 'negative'
    if 'neu' in l:
        return 'neutral'
    return l


def predict_in_batches(texts, batch_size=32):
    preds = []
    id2label = sentiment_pipe.model.config.id2label

    for i in tqdm(range(0, len(texts), batch_size), desc='Running inference'):
        batch = texts[i:i + batch_size]
        outputs = sentiment_pipe(batch, truncation=True, max_length=128)

        for out in outputs:
            preds.append({
                'pred_label_raw': out['label'],
                'pred_label': normalize_label(out['label'], id2label),
                'pred_score': float(out['score']),
            })
    return pd.DataFrame(preds)

In [ ]:
pred_df = predict_in_batches(df['text'].tolist(), batch_size=32)

df_pred = pd.concat([df.reset_index(drop=True), pred_df], axis=1)

print(df_pred[['pred_label_raw', 'pred_label']].head())
print('\nPredicted label distribution:')
print(df_pred['pred_label'].value_counts())

In [ ]:
# Quick evaluation against dataset labels (binary ground truth)
eval_df = df_pred[df_pred['pred_label'].isin(['positive', 'negative'])].copy()

print(f'Evaluable rows (pred in pos/neg): {len(eval_df)} / {len(df_pred)}')
print(f'Coverage: {len(eval_df) / len(df_pred) * 100:.1f}%')

if len(eval_df) > 0:
    acc = accuracy_score(eval_df['sentiment'], eval_df['pred_label'])
    print(f'\nAccuracy on evaluable rows: {acc:.4f}')
    print('\nClassification report:')
    print(classification_report(eval_df['sentiment'], eval_df['pred_label']))

In [ ]:
# Manual spot-check: read random examples to verify if predictions look sensible
sample_check = df_pred.sample(20, random_state=42)[['text', 'sentiment', 'pred_label', 'pred_score']]
pd.set_option('display.max_colwidth', 180)
sample_check

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
out_path = '../data/processed/tweets_with_sentiment.csv'
df_pred.to_csv(out_path, index=False)

print('Saved:', out_path)
print('Columns:', df_pred.columns.tolist())